# Part 17 (실습) — 배포와 신뢰성: 프로덕션 요소 적용

> **이 노트북의 목표**
> "한 번 되는 코드"를 "안정적으로 도는 코드"로 바꿈. `InMemorySaver`→`SqliteSaver`(영속), `max_retries`·`timeout`(재시도), 도구 예외 처리, `recursion_limit`(무한 루프 방어)를 직접 적용함.
>
> **사용 모델**: `gemini-3-flash` · **선행**: Part 0~16

## 0. 준비

In [ ]:
%pip install -q -U langchain langchain-google-genai langgraph langgraph-checkpoint-sqlite

In [ ]:
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
print("준비 완료")

## 1. 오류 처리 — 재시도·타임아웃

모델 호출은 일시적으로 실패할 수 있음(rate limit·네트워크). `max_retries`·`timeout`으로 견딤.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# 프로토타입
proto = ChatGoogleGenerativeAI(model="gemini-3-flash")

# 프로덕션 — 일시적 실패에 견디게
model = ChatGoogleGenerativeAI(
    model="gemini-3-flash",
    max_retries=3,      # 실패 시 자동 재시도 (지수 백오프)
    timeout=30,         # 30초 내 응답 없으면 중단
)
print("모델에 재시도·타임아웃 설정 완료")

## 2. 도구 예외 처리 — 우아한 저하

도구가 실패해도 터지지 않게, 예외를 잡아 모델이 읽을 메시지로 돌려줌.

In [ ]:
from langchain_core.tools import tool

@tool
def fetch_data(query: str) -> str:
    """외부 API에서 데이터를 조회한다. 데이터가 필요할 때 사용한다."""
    try:
        # 실제론 외부 API 호출. 여기선 'fail'이 들어오면 실패 시뮬레이션
        if "fail" in query:
            raise TimeoutError("외부 API 응답 없음")
        return f"'{query}' 조회 결과: 정상 데이터"
    except TimeoutError as e:
        # 예외를 모델이 이해할 메시지로 변환 (에이전트가 대응 판단)
        return f"⚠️ 조회 실패({e}). 잠시 후 다시 시도하거나 다른 방법을 안내하세요."

print("정상:", fetch_data.invoke({"query": "매출"}))
print("실패:", fetch_data.invoke({"query": "fail_test"}))
print("→ 실패해도 예외로 터지지 않고, 모델이 읽고 대응할 메시지를 돌려줌")

## 3. 영속 상태 — InMemory → SQLite

`InMemorySaver`는 재시작 시 소실. `SqliteSaver`는 DB 파일에 저장해 재시작에도 유지됨. (교체는 거의 한 줄)

In [ ]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    messages: Annotated[list, operator.add]

def chat_node(state: State) -> dict:
    last = state["messages"][-1]
    reply = model.invoke(f"사용자: {last}\n간결히 답해.").content
    return {"messages": [f"AI: {reply}"]}

builder = StateGraph(State)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# 영속 체크포인터 — SQLite 파일에 저장
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

conn = sqlite3.connect("agent_state.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)
graph = builder.compile(checkpointer=checkpointer)
print("SqliteSaver 장착 — 상태가 agent_state.db 파일에 저장됨 (재시작에도 유지)")

In [ ]:
# 대화 저장
cfg = {"configurable": {"thread_id": "user-1"}}
graph.invoke({"messages": ["내 이름은 춘현이야"]}, config=cfg)
graph.invoke({"messages": ["내 이름 기억해?"]}, config=cfg)

# 상태가 DB 파일에 저장됨 → 새 graph 객체(재시작 흉내)로도 같은 thread를 이어갈 수 있음
graph2 = builder.compile(checkpointer=SqliteSaver(conn))   # "재시작" 후 다시 연결
state = graph2.get_state(cfg)
print("재연결 후에도 남아있는 메시지 수:", len(state.values["messages"]))
print("→ InMemory였다면 재시작 시 사라졌을 대화가, SQLite엔 그대로 유지됨")

## 4. 안전장치 — recursion_limit

무한 루프를 막는 차단기. 최대 단계를 넘으면 `GraphRecursionError`가 나고, 이를 잡아 우아하게 처리함.

In [ ]:
from langgraph.errors import GraphRecursionError

# 일부러 끝나지 않는 루프 그래프
class LoopState(TypedDict):
    n: int

def forever(state: LoopState) -> dict:
    return {"n": state["n"] + 1}   # 종료 조건 없이 계속 증가

lb = StateGraph(LoopState)
lb.add_node("loop", forever)
lb.add_edge(START, "loop")
lb.add_edge("loop", "loop")        # 자기 자신으로 — 무한 루프!
loop_graph = lb.compile()

# recursion_limit으로 차단 + 예외 처리
try:
    loop_graph.invoke({"n": 0}, config={"recursion_limit": 10})
except GraphRecursionError:
    print("✅ recursion_limit(10) 도달 → GraphRecursionError 발생 → 차단됨")
    print("→ 이를 잡아 run을 '오류'로 표시하고 사용자에게 우아하게 안내(17-3)")

> 📌 **이중 안전장치**: `recursion_limit`(그래프 단계 상한) + 상태의 단계 카운터(Part 13의 `attempts` 제한)를 함께 두면 더 안전함. 무제한은 프로덕션 사고의 씨앗.

## 5. 프로덕션 전환 체크리스트

프로토타입 → 프로덕션, 무엇을 점검했나 정리.

In [ ]:
checklist = {
    "영속 상태": "InMemorySaver → SqliteSaver/PostgresSaver (재시작에도 유지) ✅",
    "재시도·타임아웃": "model에 max_retries, timeout 설정 ✅",
    "도구 예외 처리": "try/except로 잡아 모델이 읽을 메시지로 변환 ✅",
    "무한 루프 방어": "recursion_limit + 상태 단계 카운터 ✅",
    "비용·지연": "모델 차등·메시지 트리밍·캐싱 (Part 17-5) 🔲",
    "위험 행동 가드": "비가역 행동에 HITL (Part 14) 🔲",
    "관측·평가": "트레이싱·평가셋·모니터링 (Part 16) 🔲",
}
for k, v in checklist.items():
    print(f"• {k}: {v}")

## 정리

| 절 | 적용한 것 | API |
|---|---|---|
| 1 | 재시도·타임아웃 | `max_retries`, `timeout` |
| 2 | 도구 예외 처리 | try/except → 메시지 |
| 3 | 영속 상태 | `SqliteSaver` |
| 4 | 무한 루프 방어 | `recursion_limit` |
| 5 | 체크리스트 | 프로덕션 전환 |

### 3줄 요약
1. 영속 체크포인터로 재시작에도 상태 유지(교체는 거의 한 줄).
2. 재시도·타임아웃·예외 처리로 실패에 견디고, recursion_limit으로 폭주 방어.
3. "한 번 됨"과 "항상 안정"은 다른 문제 — 운영 요소를 더함.

### ▶ 다음 (Part 18 · 최종 프로젝트)
체인부터 그래프·HITL·멀티에이전트·관측·신뢰성까지 — 배운 전부를 결합한 완성형 시스템을 만들고 졸업.